# Information
## How to
1. Set the parameters. **UPPERCASE** letters are user input variables
2. Run the reprojection cell


# Requirements

In [1]:
# Standard libraries
import os
import rasterio
import numpy as np

from pathlib import Path
from importlib_resources import files

# Custom modules
from beak.experimental.utilities.preparation import impute_data
from beak.experimental.utilities.io import create_file_list, save_raster, check_path


# Scaling

**Scaling** all numerical folders within a specified model configuration.<br>
Reads the <code>ROOT_FOLDER</code> and takes the <code>NUMERICAL</code> subfolder within each model configuration.

**User inputs**

In [2]:
BASE_PATH = files("beak.data")
BASE_NAME = Path("MCCAFFERTY23") / "PROCESSED"
BASE_SPATIAL = "EPSG_4326_RES_0_025"
BASE_EXTENT = "US_CANADA"

# Set input folder
PATH_INPUT = BASE_PATH / BASE_NAME / BASE_SPATIAL / BASE_EXTENT / "GEOPHYSICS"

# Export to current location for testing purposes
CURRENT_DIR = Path(os.getcwd())
PATH_EXPORT = CURRENT_DIR / "PROCESSED" / BASE_SPATIAL / BASE_EXTENT / "GEOPHYSICS"

# Set base raster: In this case, we use the base raster created with the Lawley (2022) datacube
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / (BASE_SPATIAL + "_" + BASE_EXTENT + ".tif")
  
print(f"Input folder: {PATH_INPUT}")    
print(f"Export folder: {PATH_EXPORT}")

Input folder: s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\MCCAFFERTY23\PROCESSED\EPSG_4326_RES_0_025\US_CANADA\GEOPHYSICS
Export folder: s:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\examples\notebooks\dataset_mccafferty23\PROCESSED\EPSG_4326_RES_0_025\US_CANADA\GEOPHYSICS


In [3]:
file_list = create_file_list(PATH_INPUT, recursive=False)
print(f"Found {len(file_list)} files in the input folder")

Found 16 files in the input folder


**Run**

In [4]:
output_folder = PATH_EXPORT / str(PATH_INPUT.name + "_IMPUTED")

base_raster = rasterio.open(BASE_RASTER)
base_meta = base_raster.meta.copy()
base_raster = base_raster.read()

for file in file_list:
  raster = rasterio.open(file)
  array = raster.read()

  # Impute data
  print(f"Imputing data for {file}...")
  imputed_array = np.where(array == raster.nodata, np.nan, array)
  imputed_array = impute_data(imputed_array)
  imputed_array = np.where(base_raster == base_meta["nodata"], np.nan, imputed_array)
  
  out_path = output_folder / Path(file).name
  check_path(Path(os.path.dirname(out_path)))
  
  # Save
  save_raster(out_path, array=imputed_array, dtype="float32", metadata=raster.meta)   


Imputing data for s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\MCCAFFERTY23\PROCESSED\EPSG_4326_RES_0_025\US_CANADA\GEOPHYSICS\Gravity.tif...
Imputing data for s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\MCCAFFERTY23\PROCESSED\EPSG_4326_RES_0_025\US_CANADA\GEOPHYSICS\Gravity_HGM.tif...
Imputing data for s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\MCCAFFERTY23\PROCESSED\EPSG_4326_RES_0_025\US_CANADA\GEOPHYSICS\Gravity_Up30km.tif...
Imputing data for s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\MCCAFFERTY23\PROCESSED\EPSG_4326_RES_0_025\US_CANADA\GEOPHYSICS\Gravity_Up30km_HGM.tif...
Imputing data for s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\MCCAFFERTY23\PROCESSED\EPSG_4326_RES_0_025\US_CANADA\GEOPHYSICS\LAB.tif...
Imputing data for s:\projekte\20230082_darpa_criticalmaas_ta3\bearb